# LED Manager Smoke Tests

Staged tests for `LedSource` and `LedManager`. The physical setup combines SyncBoard and ASI Tiger LED sources in one manager, which routes each logical LED to its owning controller.

In [15]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys
import time


def find_repo_root() -> Path:
    """Return the repository root containing the evomachine package."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "evomachine" / "peripherals" / "leds.py").is_file():
            return candidate
    raise RuntimeError("Could not find the EvoMachine repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evomachine.bindings.binding_types import BindingType
from evomachine.peripherals.leds import LedConfig, LedFactory, LedManager
from evomachine.peripherals.peripheralcontrollers import (
    PeripheralControllerConfig,
    PeripheralControllerFactory,
    SerialPeripheralControllerConfig,
)
from evomachine.types import LEDType


@dataclass(frozen=True, kw_only=True)
class LedManagerTestSettings:
    binding: BindingType 
    available_leds: tuple[LEDType, ...]
    test_led: LEDType 
    test_brightness: float
    test_duration_ms: float 
    port: str | None = None
    hwid: str | None = None

syncboard_test_settings = LedManagerTestSettings(
    binding=BindingType.SYNCBOARD,
    available_leds=(LEDType.LED_385_NM, LEDType.LED_450_NM, LEDType.LED_515_NM, LEDType.LED_565_NM, LEDType.LED_645_NM),
    test_led=LEDType.LED_450_NM,
    test_brightness=100.0,
    test_duration_ms=100.0,
    hwid = "USB VID:PID=16C0:0483 SER=14582700 LOCATION=7-2:1.0",
)
asi_tiger_test_settings = LedManagerTestSettings(
    binding=BindingType.ASI_TIGER,
    available_leds=(LEDType.LED_OVERHEAD_TIGER,),
    test_led=LEDType.LED_OVERHEAD_TIGER,
    test_brightness=100.0,
    test_duration_ms=100.0,
    hwid = "USB VID:PID=10C4:EA60 SER=0001 LOCATION=5-3",
)
#Not sure this is connected
kwr103_test_settings = LedManagerTestSettings(
    binding=BindingType.KWR103,
    available_leds=(LEDType.LED_OVERHEAD,),
    test_led=LEDType.LED_OVERHEAD,
    test_brightness=100.0,
    test_duration_ms=100.0
    
)

# Physical examples:
# SyncBoard: binding=BindingType.SYNCBOARD, available_leds=(LEDType.LED_385_NM,LEDType.LED_450_NM,LEDType.LED_515_NM,LEDType.LED_565_NM,LEDType.LED_645_NM,), hwid="USB VID:PID=16C0:0483 SER=14582700 LOCATION=7-2:1.0"
# ASI Tiger: binding=BindingType.ASI_TIGER, available_leds=(LEDType.LED_OVERHEAD_TIGER,), hwid="USB VID:PID=10C4:EA60 SER=0001 LOCATION=5-3"
# KWR103: binding=BindingType.KWR103, available_leds=(LEDType.LED_OVERHEAD,), port="/dev/ttyUSB0"

SETTINGS: tuple[LedManagerTestSettings, ...] = (
    syncboard_test_settings,
    asi_tiger_test_settings,
)
TEST_SETTINGS = asi_tiger_test_settings  # Selects the LED used by the single-pulse cell.

# This must be changed deliberately before either illumination cell will run.
RUN_LED_TEST = True
SETTINGS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


(LedManagerTestSettings(binding=<BindingType.SYNCBOARD: 3>, available_leds=(<LEDType.LED_385_NM: 0>, <LEDType.LED_450_NM: 1>, <LEDType.LED_515_NM: 2>, <LEDType.LED_565_NM: 3>, <LEDType.LED_645_NM: 4>), test_led=<LEDType.LED_450_NM: 1>, test_brightness=100.0, test_duration_ms=100.0, port=None, hwid='USB VID:PID=16C0:0483 SER=14582700 LOCATION=7-2:1.0'),
 LedManagerTestSettings(binding=<BindingType.ASI_TIGER: 2>, available_leds=(<LEDType.LED_OVERHEAD_TIGER: 6>,), test_led=<LEDType.LED_OVERHEAD_TIGER: 6>, test_brightness=100.0, test_duration_ms=100.0, port=None, hwid='USB VID:PID=10C4:EA60 SER=0001 LOCATION=5-3'))

## Inspect serial ports

Use this to identify each physical controller. Set exactly one of `port` or `hwid` on each settings object; neither is needed for a virtual binding.

In [14]:
try:
    from serial.tools import list_ports
except ImportError as error:
    raise RuntimeError("pyserial is required to inspect serial ports.") from error

[
    {"device": port.device, "description": port.description, "hwid": port.hwid}
    for port in list_ports.comports()
]

[{'device': '/dev/ttyS1', 'description': 'ttyS1', 'hwid': 'PNP0501'},
 {'device': '/dev/ttyS0', 'description': 'ttyS0', 'hwid': 'PNP0501'},
 {'device': '/dev/ttyUSB0',
  'description': 'CP2102 USB to UART Bridge Controller - CP2102 USB to UART Bridge Controller',
  'hwid': 'USB VID:PID=10C4:EA60 SER=0001 LOCATION=5-3'},
 {'device': '/dev/ttyACM0',
  'description': 'USB Serial',
  'hwid': 'USB VID:PID=16C0:0483 SER=14582700 LOCATION=7-2:1.0'}]

## Create both controllers, sources, and one manager

Initialisation should leave every LED disabled. Re-running this cell cleans up objects from its previous run first.

In [16]:
previous_manager = globals().get("led_manager")
if previous_manager is not None and previous_manager.is_initialised():
    previous_manager.disable_led()
    previous_manager.finalise()
for previous_controller in globals().get("led_controllers", []):
    if previous_controller.is_initialised():
        previous_controller.shutdown()

controller_configs = []
for settings in SETTINGS:
    if settings.binding == BindingType.VIRTUAL:
        controller_configs.append(PeripheralControllerConfig(binding=BindingType.VIRTUAL))
    elif settings.binding in {BindingType.SYNCBOARD, BindingType.ASI_TIGER, BindingType.KWR103}:
        controller_configs.append(
            SerialPeripheralControllerConfig(
                binding=settings.binding,
                port=settings.port,
                hwid=settings.hwid,
            )
        )
    else:
        raise ValueError(f"Unsupported LED binding: {settings.binding}")

led_controllers = [PeripheralControllerFactory.create(config) for config in controller_configs]
led_sources = [
    LedFactory.create(
        LedConfig(binding=settings.binding, available_leds=list(settings.available_leds)),
        peripheral_controllers=led_controllers,
    )
    for settings in SETTINGS
]
led_manager = LedManager(led_sources)
led_manager.initialise()
led_manager.disable_led()

{
    "manager": led_manager.name,
    "sources": [source.name for source in led_sources],
    "bindings": [settings.binding for settings in SETTINGS],
    "controllers_initialised": [controller.is_initialised() for controller in led_controllers],
    "manager_initialised": led_manager.is_initialised(),
    "manager_alive": led_manager.is_alive(),
    "available_leds": led_manager.get_available_leds(),
}

2026-07-02 11:32:29 - DEBUG - syncboard.serialconnection - Connecting to /dev/ttyACM0 at 2000000 baud
2026-07-02 11:32:29 - DEBUG - syncboard.command - Formatted command: $attachLED/true#%
2026-07-02 11:32:29 - DEBUG - syncboard.command - Formatted command: $attachLED/true#%
2026-07-02 11:32:29 - DEBUG - syncboard.syncboardcontroller - Sending command $attachLED/true#% at attempt 0.
2026-07-02 11:32:29 - DEBUG - syncboard.syncboardcontroller - Sending command $attachLED/true#% at attempt 0.
2026-07-02 11:32:29 - DEBUG - syncboard.serialconnection - Sending data: b'$attachLED/true#%'
2026-07-02 11:32:29 - DEBUG - syncboard.serialconnection - Sending data: b'$attachLED/true#%'
2026-07-02 11:32:29 - DEBUG - syncboard.serialconnection - Received: $error/Cant attach LED while system is enabled#%

2026-07-02 11:32:29 - DEBUG - syncboard.serialconnection - Received: $error/Cant attach LED while system is enabled#%

2026-07-02 11:32:29 - DEBUG - syncboard.serialconnection - Received: ['$error/

{'manager': 'LED Manager',
 'sources': ['SyncBoard LED Source', 'ASI Tiger LED Source'],
 'bindings': [<BindingType.SYNCBOARD: 3>, <BindingType.ASI_TIGER: 2>],
 'controllers_initialised': [True, True],
 'manager_initialised': True,
 'manager_alive': True,
 'available_leds': [<LEDType.LED_385_NM: 0>,
  <LEDType.LED_450_NM: 1>,
  <LEDType.LED_515_NM: 2>,
  <LEDType.LED_565_NM: 3>,
  <LEDType.LED_645_NM: 4>,
  <LEDType.LED_OVERHEAD_TIGER: 6>]}

## Inspect cached LED states

In [3]:
if "led_manager" not in globals() or not led_manager.is_initialised():
    raise RuntimeError("Run the manager creation cell first.")

[led_manager.get_led_state(led_type) for led_type in led_manager.get_available_leds()]

[LedState(led_type=<LEDType.LED_385_NM: 0>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_450_NM: 1>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_515_NM: 2>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_565_NM: 3>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_645_NM: 4>, brightness=0.0, is_on=False, stop_time=None)]

## Test one timed LED pulse

Confirm that `TEST_SETTINGS.test_led` identifies the intended physical channel. The `finally` block disables LEDs on both controllers even if the test raises an exception.

In [24]:
if not RUN_LED_TEST:
    raise RuntimeError("Set RUN_LED_TEST = True in the setup cell after checking the hardware configuration.")
if TEST_SETTINGS.test_led not in led_manager.get_available_leds():
    raise ValueError(f"{TEST_SETTINGS.test_led} is not managed by this LED manager.")

try:
    led_manager.set_led(
        led_type= LEDType.LED_OVERHEAD_TIGER, #TEST_SETTINGS.test_led,
        brightness=TEST_SETTINGS.test_brightness,
        duration=TEST_SETTINGS.test_duration_ms*100,
    )
    state_during_pulse = led_manager.get_led_state(TEST_SETTINGS.test_led)
finally:
    led_manager.disable_led()

{
    "during_pulse": state_during_pulse,
    "after_disable": led_manager.get_led_state(TEST_SETTINGS.test_led),
}

2026-07-02 11:35:54 - DEBUG - evomachine.peripherals.leds - LedManager.set_led: routing LED_OVERHEAD_TIGER brightness=100.0 duration=10000.0 through LED Manager.
2026-07-02 11:35:54 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 11:35:54 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 11:35:54 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 11:35:54 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 11:35:54 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting ASI Tiger LED Source LED_OVERHEAD_TIGER to brightness=100.0 duration=10000.0.
2026-07-02 11:35:54 - DEBUG - asitiger.serialconnection - Sending data: b'7LED Y=100\r'
2026-07-02 11:35:54 - DEBUG - asitiger.serialconnection - Sending data: b'7LED Y=100\r'
2026-07-02 11:35:54 - DEBUG - asitiger.serialconnection - Received: b':A\r\n'
2026-07-02 11:35:54 - DEBUG - asitiger.serialconnection - Received: b':A\r\n'
2026-07-02 11:35:54 - DEBUG -

{'during_pulse': LedState(led_type=<LEDType.LED_OVERHEAD_TIGER: 6>, brightness=100.0, is_on=True, stop_time=1782988564.4968977),
 'after_disable': LedState(led_type=<LEDType.LED_OVERHEAD_TIGER: 6>, brightness=0.0, is_on=False, stop_time=None)}

## Test manager routing across all configured LEDs

Each channel receives one short pulse in sequence. Skip this cell unless every configured channel is safe to illuminate.

In [21]:
if not RUN_LED_TEST:
    raise RuntimeError("Set RUN_LED_TEST = True in the setup cell after checking the hardware configuration.")

settings_by_led = {
    led_type: settings
    for settings in SETTINGS
    for led_type in settings.available_leds
}
try:
    for led_type in led_manager.get_available_leds():
        settings = settings_by_led[led_type]
        print(f"Pulsing {led_type.name}")
        led_manager.set_led(
            led_type=led_type,
            brightness=settings.test_brightness,
            duration=settings.test_duration_ms*10,
        )
        # time.sleep(settings.test_duration_ms / 1000.0 + 0.1)
finally:
    led_manager.disable_led()

[led_manager.get_led_state(led_type) for led_type in led_manager.get_available_leds()]

2026-07-02 11:34:00 - DEBUG - evomachine.peripherals.leds - LedManager.set_led: routing LED_385_NM brightness=100.0 duration=1000.0 through LED Manager.
2026-07-02 11:34:00 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting SyncBoard LED Source LED_385_NM to brightness=100.0 duration=1000.0.
2026-07-02 11:34:00 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is 1000.0.
2026-07-02 11:34:00 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is 1000.0.
2026-07-02 11:34:00 - DEBUG - syncboard.syncboardcontroller - LED LED_ID.LED_385_NM already configured: {'mode': 0, 'intensity': 1.0, 'status': 'off', 'stop_time': 1782988416.1853774}
2026-07-02 11:34:00 - DEBUG - syncboard.syncboardcontroller - LED LED_ID.LED_385_NM already configured: {'mode': 0, 'intensity': 1.0, 'status': 'off', 'stop_time': 1782988416.1853774}
2026-07-02 11:34:00 - DEBUG - syncboard.command - Formatted command: $switchL

Pulsing LED_385_NM


2026-07-02 11:34:01 - DEBUG - syncboard.serialconnection - Received: Turning LED 7 off after 1000008 microseconds.

2026-07-02 11:34:01 - DEBUG - syncboard.serialconnection - Received: Turning LED 7 off after 1000008 microseconds.

2026-07-02 11:34:01 - DEBUG - evomachine.peripherals.leds - LedManager.set_led: routing LED_450_NM brightness=100.0 duration=1000.0 through LED Manager.
2026-07-02 11:34:01 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting SyncBoard LED Source LED_450_NM to brightness=100.0 duration=1000.0.
2026-07-02 11:34:01 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is 1000.0.
2026-07-02 11:34:01 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is 1000.0.
2026-07-02 11:34:01 - DEBUG - syncboard.syncboardcontroller - LED LED_ID.LED_450_NM already configured: {'mode': 0, 'intensity': 1.0, 'status': 'off', 'stop_time': 1782988416.2897115}
2026-07-02 11:34:01 - DEBUG -

Pulsing LED_450_NM


2026-07-02 11:34:02 - DEBUG - syncboard.serialconnection - Received: Turning LED 5 off after 1000008 microseconds.

2026-07-02 11:34:02 - DEBUG - syncboard.serialconnection - Received: Turning LED 5 off after 1000008 microseconds.

2026-07-02 11:34:02 - DEBUG - evomachine.peripherals.leds - LedManager.set_led: routing LED_515_NM brightness=100.0 duration=1000.0 through LED Manager.
2026-07-02 11:34:02 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting SyncBoard LED Source LED_515_NM to brightness=100.0 duration=1000.0.
2026-07-02 11:34:02 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is 1000.0.
2026-07-02 11:34:02 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is 1000.0.
2026-07-02 11:34:02 - DEBUG - syncboard.syncboardcontroller - LED LED_ID.LED_515_NM already configured: {'mode': 0, 'intensity': 1.0, 'status': 'off', 'stop_time': 1782988416.395414}
2026-07-02 11:34:02 - DEBUG - 

...

Pulsing LED_645_NM


2026-07-02 11:34:05 - DEBUG - syncboard.serialconnection - Received: Turning LED 4 off after 1000008 microseconds.

2026-07-02 11:34:05 - DEBUG - syncboard.serialconnection - Received: Turning LED 4 off after 1000008 microseconds.

2026-07-02 11:34:05 - DEBUG - evomachine.peripherals.leds - LedManager.set_led: routing LED_OVERHEAD_TIGER brightness=100.0 duration=1000.0 through LED Manager.
2026-07-02 11:34:05 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 11:34:05 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 11:34:05 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 11:34:05 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 11:34:05 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting ASI Tiger LED Source LED_OVERHEAD_TIGER to brightness=100.0 duration=1000.0.
2026-07-02 11:34:05 - DEBUG - asitiger.serialconnection - Sending data: b'7LED Y=100\r'
2026-07-02 11:34:05 - DEBUG - asitiger.seri

Pulsing LED_OVERHEAD_TIGER


2026-07-02 11:34:06 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 11:34:06 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 11:34:06 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 11:34:06 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 11:34:06 - DEBUG - evomachine.peripherals.leds - LedSource.disable_led: disabling LED_OVERHEAD_TIGER on ASI Tiger LED Source.
2026-07-02 11:34:06 - DEBUG - asitiger.serialconnection - Sending data: b'7LED Y=0\r'
2026-07-02 11:34:06 - DEBUG - asitiger.serialconnection - Sending data: b'7LED Y=0\r'
2026-07-02 11:34:06 - DEBUG - asitiger.serialconnection - Received: b':A\r\n'
2026-07-02 11:34:06 - DEBUG - asitiger.serialconnection - Received: b':A\r\n'
2026-07-02 11:34:06 - DEBUG - syncboard.serialconnection - Received: 
2026-07-02 11:34:06 - DEBUG - syncboard.serialconnection - Received: 
2026-07-02 11:34:06 - DEBUG - evomachine.peripherals.leds - LedSource.disable_led: 

[LedState(led_type=<LEDType.LED_385_NM: 0>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_450_NM: 1>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_515_NM: 2>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_565_NM: 3>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_645_NM: 4>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_OVERHEAD_TIGER: 6>, brightness=0.0, is_on=False, stop_time=None)]

## Cleanup

Always run this before unplugging or switching LED hardware.

In [1]:
if "led_manager" in globals() and led_manager is not None:
    led_manager.disable_led()
    if led_manager.is_initialised():
        led_manager.finalise()

for led_controller in globals().get("led_controllers", []):
    if led_controller.is_initialised():
        led_controller.shutdown()

{
    "manager_initialised": led_manager.is_initialised(),
    "controllers_initialised": [
        controller.is_initialised() for controller in globals().get("led_controllers", [])
    ],
}

NameError: name 'led_manager' is not defined